## Preprocesamiento y homologación de la Muestra 3

Con el objetivo de estandarizar la información y facilitar la comparación de los registros de la **Muestra 3** con el catálogo vehicular, se realizó un proceso de homologación de variables orientado a corregir desplazamientos estructurales (desfases), uniformar codificaciones y depurar descripciones.

Para preservar la información original, las transformaciones se efectuaron sobre una copia de la base de datos, denominada `muestra_3_homologada`, utilizando un `respaldo_control` inmutable, siguiendo el orden que se describe a continuación.

### 1. Identificación y corrección de desfases

Inicialmente, se detectó y corrigió un problema de alineación en las columnas que afectaba a un total de 67 registros, el cual se abordó considerando dos procedimientos:

1. **Desfase a la derecha (7 registros):** La variable `Ocupantes` contenía valores de texto correspondientes a la transmisión (ej. `PDK AUT.`, `AUT.`, `STD.`) y desplazaba el año del vehículo. Se reubicó el texto de la transmisión al final de la `descripcion`, trasladando el año a su variable original, mientras que `Modelo` se vació temporalmente para su ajuste lógico.
2. **Desfase a la izquierda (60 registros):** La cantidad de ocupantes quedó anidada dentro de la `descripcion` (ej. `5 OCUP.`), desplazando el año hacia la variable `Ocupantes` y dejando a `Modelo` con valores nulos. Se extrajo la cantidad numérica mediante expresiones regulares para rellenar los ocupantes, se eliminó dicho fragmento de la descripción y se reubicó el año en la columna `Modelo`.

### 2. Homologación de ocupantes y modelo

Una vez corregidos los desplazamientos, se estandarizaron las variables numéricas `Ocupantes` y `Modelo`. 

En el caso de los ocupantes, se extrajeron únicamente los dígitos, omitiendo prefijos o sufijos. Posteriormente, ambas variables fueron convertidas a formato numérico entero limpio (`Int64`), eliminando valores flotantes no deseados.

| Variable original | Formato antes del ajuste | Formato homologado (`Int64`) |
|---|---|---|
| Ocupantes (desfasado) | 5 OCUP. (en descripción) | 5 |
| Modelo (desfasado) | En columna Ocupantes | Año en numérico |

Esta transformación permitió unificar la representación de los años y la capacidad de pasajeros con el formato numérico esperado para su comparación.

### 3. Creación y homologación de transmisión

Dado que la Muestra 3 no contaba con una columna explícita para la transmisión, se procedió a extraer esta información a partir de la variable `descripcion` para alinearla con el catálogo vehicular.

El procedimiento se desarrolló en el siguiente orden:

1. **Extracción de terminaciones:** Se aisló la última palabra de cada descripción, ignorando signos de puntuación, para identificar los términos asociados a la transmisión del vehículo.
2. **Homologación de transmisión:** Se transformaron los términos extraídos a la nomenclatura utilizada en el catálogo vehicular, creando la nueva variable `Transmisión`:

| Patrones identificados | Código homologado | Significado |
|---|---|---|
| AUT, PDK, TIPTRONIC, CVT, SELECTSHIFT, 7SPEED | A | Transmisión automática |
| STD, MAN | S | Transmisión estándar o manual |

Los registros cuya última palabra no coincidió con ninguno de los términos de referencia se mantuvieron catalogados como valores faltantes (`<NA>`).

### 4. Limpieza de descripciones

La última etapa consistió en estandarizar la información contenida en la variable `descripcion`.

La limpieza se enfocó en eliminar el punto final (`.`) ubicado al cierre de cada registro. Adicionalmente, se eliminaron los espacios en blanco sobrantes al inicio y al final de las cadenas de texto, conservando intactos los acrónimos de transmisión (como `AUT` o `STD`) que formaban parte del texto original o que fueron concatenados tras la corrección del desfase.

### Alcance de las transformaciones

Las homologaciones se limitaron a las correcciones de estructura (desfases) plenamente identificadas mediante reglas lógicas y a las equivalencias establecidas en la creación de la variable de transmisión. En particular, el mapeo de la `Transmisión` presupone que el último fragmento de texto en la descripción corresponde invariablemente al tipo de caja del vehículo; los valores que no cumplieron esta condición requieren revisión.

Por otra parte, no se efectuaron procedimientos generales de imputación ni eliminación de registros por valores faltantes en ninguna de las variables numéricas (`Modelo`, `Ocupantes`) ni categóricas. Estos se conservaron en la base resultante.

Por lo tanto, el procedimiento **no garantiza la correspondencia exacta de todos los registros con el catálogo vehicular**, sino que constituye una etapa de preparación y estandarización de los datos para su posterior comparación y asignación de claves.